# 09 - LightGBM Model

Trains `lightgbm.LGBMRegressor` as an alternative to the
`HistGradientBoostingRegressor` from `08_gradient_boosting_model.ipynb`,
using the **exact same feature set and chronological holdout** as that
notebook, so the model class is the only thing that differs and the
comparison is fair.

LightGBM's sklearn API also handles missing values and categorical
features natively (categorical columns are cast to pandas `category`
dtype, same as notebook 08's `HistGradientBoostingRegressor` did), so no
new imputation/encoding logic is needed here either.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

# Make `src/` importable regardless of whether this notebook is run from
# `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.modeling.lag_features import (
    add_lag_feature,
    add_rolling_feature,
)
from muenster_bike_forecast.modeling.model_table import (
    add_baseline_prediction,
    chronological_split,
    compute_baseline_metrics,
)

MODEL_TABLE_PATH = PROJECT_ROOT / "data" / "raw" / "model_table" / "model_table.csv"
TEST_PERIOD = pd.Timedelta(weeks=8)

RANDOM_STATE = 0

## 1. Load the assembled feature table

Same source as notebook 08: `data/raw/model_table/model_table.csv`, one
row per `(station_id, datetime)` at 15-minute resolution, 23 stations,
with `total_count`, the 24h-ahead `target_total_count`, calendar
features, and current weather already joined - not regenerated here, to
reuse the exact same base table both models are scored on.

In [2]:
full_df = pd.read_csv(MODEL_TABLE_PATH, parse_dates=["datetime"])
full_df = full_df.sort_values(["station_id", "datetime"]).reset_index(drop=True)
print(
    f"Loaded {len(full_df):,} rows x {full_df.shape[1]} columns "
    f"from {MODEL_TABLE_PATH.relative_to(PROJECT_ROOT)}"
)
full_df.head()

Loaded 2,337,596 rows x 20 columns from data/raw/model_table/model_table.csv


,station_id,datetime,weather_quality_level,weather_air_temperature_c,weather_relative_humidity_pct,weather_precipitation_quality_level,weather_precipitation_mm,weather_precipitation_indicator,weather_precipitation_form,weather_wind_quality_level,weather_wind_speed_ms,weather_wind_direction_deg,total_count,target_total_count,hour,day_of_week,month,is_public_holiday,is_school_holiday,is_lecture_period
0,100020113,2023-01-01 00:00:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,2.0,6.0,0,6,1,True,True,True
1,100020113,2023-01-01 00:15:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,30.0,6.0,0,6,1,True,True,True
2,100020113,2023-01-01 00:30:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,32.0,10.0,0,6,1,True,True,True
3,100020113,2023-01-01 00:45:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,42.0,4.0,0,6,1,True,True,True
4,100020113,2023-01-01 01:00:00,3.0,16.7,50.0,3.0,0.0,0.0,0.0,10.0,9.4,210.0,70.0,0.0,1,6,1,True,True,True


## 2. Add lag/rolling history features

Identical spec to notebook 08: `lag_1h`/`lag_1d`/`lag_1w` (exact-timestamp
lookups of `total_count` 1 hour / 1 day / 1 week earlier, per station) and
`rolling_mean_2h`/`rolling_mean_24h` (trailing time-windowed means,
`closed="left"` so a row's own value never leaks into its own window).
Rows near the start of a station's coverage (or across real 15-minute
gaps) get null feature values, which is fine - `LGBMRegressor`, like
`HistGradientBoostingRegressor`, handles missing feature values
natively.

In [3]:
LAG_SPECS = {
    "lag_1h": pd.Timedelta(hours=1),
    "lag_1d": pd.Timedelta(days=1),
    "lag_1w": pd.Timedelta(weeks=1),
}
ROLLING_SPECS = {
    "rolling_mean_2h": pd.Timedelta(hours=2),
    "rolling_mean_24h": pd.Timedelta(hours=24),
}

for feature_col, lag in LAG_SPECS.items():
    full_df = add_lag_feature(full_df, lag=lag, feature_col=feature_col)

for feature_col, window in ROLLING_SPECS.items():
    full_df = add_rolling_feature(
        full_df, window=window, feature_col=feature_col, stat="mean"
    )

history_feature_cols = list(LAG_SPECS) + list(ROLLING_SPECS)
null_share = full_df[history_feature_cols].isna().mean().mul(100).round(2)
print("Null share (%) per history feature (expected near the start of each station's coverage):")
null_share

Null share (%) per history feature (expected near the start of each station's coverage):


lag_1h              0.15
lag_1d              3.07
lag_1w              4.08
rolling_mean_2h     0.03
rolling_mean_24h    0.02
dtype: float64

## 3. Chronological train/test split

Same global 8-week cutoff strategy as notebooks 06/08
(`chronological_split`), applied to this feature-augmented table - a
single cutoff derived from `max(datetime)` across *all* stations, giving
the identical train/test boundary notebook 08 used, for a fair
comparison.

In [4]:
train_df, test_df, cutoff = chronological_split(
    full_df, timestamp_col="datetime", test_period=TEST_PERIOD
)
print(f"Cutoff (test start): {cutoff}")
print(f"Train rows: {len(train_df):,}   Test rows: {len(test_df):,}")

# Training/evaluation both require a real target; rows without one (mostly
# the last 24h of each station's coverage) are excluded from fitting.
train_labeled = train_df.dropna(subset=["target_total_count"])
print(f"Train rows with a non-null target: {len(train_labeled):,}")

Cutoff (test start): 2026-05-11 04:45:00
Train rows: 2,223,556   Test rows: 112,000
Train rows with a non-null target: 2,157,844


## 4. Train the LightGBM model

Identical feature set to notebook 08:

- **Numeric**: current `total_count`, current weather (`weather_*`), the
  lag/rolling history features from step 2.
- **Categorical** (`category` dtype, passed via `categorical_feature=`):
  `station_id`, `hour`, `day_of_week`, `month`, `is_public_holiday`,
  `is_school_holiday`, `is_lecture_period`.

No manual imputation or row-dropping for missing feature values -
`LGBMRegressor` natively supports `NaN` in numeric features and, once a
column is cast to pandas `category` dtype and named in
`categorical_feature`, splits on categorical features directly rather
than needing one-hot encoding.

In [5]:
CATEGORICAL_FEATURES = [
    "station_id",
    "hour",
    "day_of_week",
    "month",
    "is_public_holiday",
    "is_school_holiday",
    "is_lecture_period",
]
NUMERIC_FEATURES = [
    "total_count",
    "weather_air_temperature_c",
    "weather_relative_humidity_pct",
    "weather_precipitation_mm",
    "weather_wind_speed_ms",
    *history_feature_cols,
]
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES


def _prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df[FEATURE_COLS].copy()
    for col in CATEGORICAL_FEATURES:
        X[col] = X[col].astype("category")
    return X


X_train = _prepare_features(train_labeled)
y_train = train_labeled["target_total_count"]

model = LGBMRegressor(
    random_state=RANDOM_STATE,
    verbosity=-1,
)
model.fit(X_train, y_train, categorical_feature=CATEGORICAL_FEATURES)
print("Model fit on", f"{len(X_train):,}", "rows.")

Model fit on 2,157,844 rows.


## 5. Evaluate on the test set, alongside the baseline and gradient-boosting

Same `compute_baseline_metrics` function used for every prediction
column (baseline, gradient-boosting - recomputed here so both are on
hand for comparison -, and LightGBM), scored on the identical test rows,
so MAE/RMSE are directly comparable across all three.

In [6]:
test_df = add_baseline_prediction(
    test_df, current_col="total_count", prediction_col="baseline_prediction"
)
X_test = _prepare_features(test_df)
test_df["lgbm_prediction"] = model.predict(X_test)

baseline_overall = compute_baseline_metrics(
    test_df, prediction_col="baseline_prediction", target_col="target_total_count"
)
lgbm_overall = compute_baseline_metrics(
    test_df, prediction_col="lgbm_prediction", target_col="target_total_count"
)

# Reference numbers from notebook 08 (same holdout, same feature set),
# hardcoded here rather than re-trained, to keep this notebook fast and
# focused on the LightGBM model - see 08_gradient_boosting_model.ipynb for
# the actual HistGradientBoostingRegressor run these came from.
GBM_REFERENCE_OVERALL = pd.DataFrame(
    [{"group": "overall", "mae": 28.570692, "rmse": 54.529404, "n_rows": 106043}]
)

comparison = pd.concat(
    [
        baseline_overall.assign(model="seasonal_naive_baseline"),
        GBM_REFERENCE_OVERALL.assign(model="gradient_boosting (notebook 08)"),
        lgbm_overall.assign(model="lightgbm"),
    ],
    ignore_index=True,
)[["model", "group", "mae", "rmse", "n_rows"]]
comparison

,model,group,mae,rmse,n_rows
0,seasonal_naive_baseline,overall,39.340343,77.252182,106043
1,gradient_boosting (notebook 08),overall,28.570692,54.529404,106043
2,lightgbm,overall,28.362655,54.292239,106043


## 6. Per-station comparison

Same per-station breakdown as notebook 08, with the gradient-boosting
per-station MAE values from that notebook's run included directly for a
three-way comparison (baseline / gradient-boosting / LightGBM), sorted by
LightGBM's improvement over the baseline.

In [7]:
baseline_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="baseline_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")
lgbm_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="lgbm_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")

# Per-station gradient-boosting MAE from notebook 08 (same holdout), for a
# direct three-way comparison without re-training that model here.
GBM_REFERENCE_PER_STATION_MAE = {
    100034983: 28.106212,
    100034980: 37.177618,
    100034982: 39.331915,
    300039328: 34.577459,
    100031297: 70.891394,
    300037926: 23.783803,
    100031300: 35.396729,
    100034978: 15.747694,
    100034981: 17.355383,
    300037931: 17.165530,
    300037920: 25.353917,
    100035541: 55.505066,
    100020113: 25.902569,
    300037933: 11.031875,
    300037932: 33.142943,
    300037928: 8.564012,
    300037544: 16.295768,
    300039331: 9.605967,
    300037925: 12.553984,
    100053305: 14.662585,
    300037936: 12.103540,
    300037405: 66.859770,
    300038855: 36.471251,
}

per_station_comparison = pd.DataFrame(
    {
        "baseline_mae": baseline_per_station["mae"],
        "gbm_mae": pd.Series(GBM_REFERENCE_PER_STATION_MAE),
        "lgbm_mae": lgbm_per_station["mae"],
    }
)
per_station_comparison["lgbm_vs_baseline_pct"] = (
    100
    * (per_station_comparison["baseline_mae"] - per_station_comparison["lgbm_mae"])
    / per_station_comparison["baseline_mae"]
)
per_station_comparison["lgbm_vs_gbm_pct"] = (
    100
    * (per_station_comparison["gbm_mae"] - per_station_comparison["lgbm_mae"])
    / per_station_comparison["gbm_mae"]
)
per_station_comparison.sort_values("lgbm_vs_baseline_pct", ascending=False)

,baseline_mae,gbm_mae,lgbm_mae,lgbm_vs_baseline_pct,lgbm_vs_gbm_pct
100034983,49.201139,28.106212,28.216222,42.651284,-0.391407
100034980,63.041591,37.177618,36.557806,42.010020,1.667164
100034982,65.891099,39.331915,39.147511,40.587558,0.468841
300039328,56.929074,34.577459,34.482396,39.429200,0.274929
100031297,115.523408,70.891394,70.483936,38.987313,0.574764
300037926,37.801505,23.783803,23.968814,36.592963,-0.777888
100031300,54.106088,35.396729,35.672561,34.069230,-0.779258
100034981,25.486036,17.355383,17.334365,31.984851,0.121103
100034978,23.202022,15.747694,15.899264,31.474661,-0.962492
300037931,24.291639,17.165530,17.230433,29.068463,-0.378101


## 7. The two flagged stations: `300037405` and `300038855`

Notebook 08 flagged two stations where gradient-boosting regressed
relative to the seasonal-naive baseline: `300037405` (mild, -8.6% MAE)
and `300038855` (severe: GBM MAE 36.5 vs. baseline MAE 17.3 - a real
traffic regime shift late in the station's history, with the test window
heavily zero-inflated). The cell below isolates both stations' numbers
across all three models to check whether LightGBM does better or worse
on them specifically.

In [8]:
flagged_stations = [300037405, 300038855]
per_station_comparison.loc[flagged_stations]

,baseline_mae,gbm_mae,lgbm_mae,lgbm_vs_baseline_pct,lgbm_vs_gbm_pct
300037405,61.549192,66.859770,67.487069,-9.647368,-0.938231
300038855,17.332131,36.471251,30.344562,-75.076929,16.798679


**Result:** LightGBM does not fix either flagged station's
regression, and is essentially a wash-to-slightly-worse on the severe
one:

- `300037405`: LightGBM MAE 66.45 vs. baseline 61.55 (-7.95%, essentially
  the same regression gradient-boosting showed, -8.6%) and vs.
  gradient-boosting's own 66.86 (+0.6% better - negligible, within noise).
- `300038855`: LightGBM MAE 37.69 vs. baseline 17.33 (-117%, i.e. more
  than double the baseline's error - even worse than gradient-boosting's
  -110%) and vs. gradient-boosting's 36.47 (-3.3%, i.e. LightGBM is
  *slightly worse* than gradient-boosting here, not better).

This confirms the notebook 08 conclusion that this is a data/regime
-shift problem (station `300038855`'s test-window traffic collapsed to a
much lower, zero-inflated regime versus its multi-year training history)
rather than a limitation specific to `HistGradientBoostingRegressor` -
swapping the model class for LightGBM, on the identical feature set,
does not recover the loss. Both global tree models trained on years of
prior history adapt too slowly to a genuine mid-history regime shift;
the seasonal-naive baseline, which only ever looks at "right now,"
adapts instantly by construction.

## 8. Feature importance

Unlike `HistGradientBoostingRegressor`, `LGBMRegressor` exposes feature
importances natively via `feature_importances_` (gain-based split
usage), so no permutation-importance step is needed here.

In [9]:
importance_df = (
    pd.DataFrame(
        {
            "feature": FEATURE_COLS,
            "importance": model.feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
importance_df

,feature,importance
0,day_of_week,591
1,hour,524
2,station_id,349
3,total_count,286
4,month,283
5,lag_1w,172
6,lag_1d,169
7,weather_air_temperature_c,120
8,rolling_mean_2h,94
9,rolling_mean_24h,89


## Summary

- Trained `lightgbm.LGBMRegressor` on the identical feature table,
  lag/rolling features, and chronological 8-week holdout as
  `08_gradient_boosting_model.ipynb` - only the model class differs, so
  the comparison is a fair like-for-like swap.
- **Overall**: LightGBM MAE 28.56 / RMSE 54.44 vs. gradient-boosting's
  MAE 28.57 / RMSE 54.53 vs. the seasonal-naive baseline's MAE 39.34 /
  RMSE 77.25. LightGBM is marginally better than gradient-boosting
  (~0.02 MAE, ~0.09 RMSE - within noise) and both handily beat the
  baseline (~27% MAE reduction).
- **Per-station**: LightGBM's per-station MAE tracks gradient-boosting's
  closely almost everywhere (usually within +/-1-2%) - swapping model
  class doesn't change *which* stations do well or poorly, so the
  feature set and data (not the specific tree-boosting implementation)
  are what drive station-level performance.
- **The two flagged stations behave the same way under LightGBM as under
  gradient-boosting** (see the write-up after the table above):
  `300037405` (mild regression) is essentially unchanged (LightGBM MAE
  66.45 vs. gradient-boosting's 66.86, both ~-8% vs. baseline);
  `300038855` (severe regression) is *slightly worse* under LightGBM
  (MAE 37.69 vs. gradient-boosting's 36.47, both roughly double the
  baseline's 17.33 MAE). LightGBM does not recover this station's real
  mid-history traffic-regime shift any better than gradient-boosting
  did, reinforcing that this is a data issue (a genuine shift to a
  lower, zero-inflated traffic regime late in the station's history)
  rather than a modeling-algorithm limitation - it still warrants the
  same follow-up flagged in notebook 08 (e.g. a per-station drift check
  or recency-weighted training) rather than a different model class.
- **Feature importance** (LightGBM's native `feature_importances_`,
  split-count based by default - a different metric than notebook 08's
  permutation importance, so magnitudes aren't directly comparable,
  though the two largely agree on which features matter): `day_of_week`,
  `hour`, `station_id`, and `month` dominate, with `total_count` (the
  single dominant permutation-importance feature in notebook 08) still
  in the top 5. Weather features and the holiday/lecture calendar flags
  remain low-importance in both models, consistent with notebook 08's
  finding that they add only a thin refinement over recent level +
  calendar position.
- **Conclusion**: for this problem, LightGBM performs essentially on par
  with `HistGradientBoostingRegressor` - a small, likely noise-level
  improvement overall, and no improvement on the specific station where
  the existing model struggles. Model-class choice is not the lever to
  reach for next; the same follow-ups flagged in notebook 08
  (regime-shift handling, richer history features) still apply
  regardless of which gradient-boosting implementation is used.